# 01 — Exploratory Data Analysis

MerchantShield AI inspects the ULB credit-card fraud dataset before any modeling.
The target `Class` and the timestamp `Time` are **not** used as model features.
`Time` is reserved for chronological splitting.

In [ ]:
import json
from pathlib import Path
import sys
sys.path.insert(0, str(Path('..').resolve()))

from src.config import load_config, get_project_root
from src.data.loader import load_raw_dataset, inspect_dataset
from src.data.splitter import temporal_split, split_summary
from src.data.leakage import run_leakage_checks
from src.features.preprocessing import get_feature_columns

config = load_config()
df = load_raw_dataset(config)
print(df.head())
summary = inspect_dataset(df, config)
print(json.dumps({k: summary[k] for k in ['n_rows','n_columns','n_features','target_distribution','duplicate_rows','class_imbalance_ratio']}, indent=2))

In [ ]:
train, val, test = temporal_split(
    df.drop_duplicates(),
    config['data']['time_column'],
    config['data']['train_ratio'],
    config['data']['val_ratio'],
    config['data']['test_ratio'],
)
print(split_summary({'train': train, 'validation': val, 'test': test}, 'Class'))
features = get_feature_columns(df, config['data']['exclude_columns'])
print('Leakage audit', run_leakage_checks(train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True), features, 'Class', 'Time'))